# Results Analysis. Preparing the information, plots, metrics for paper.

In [ ]:
import time
import pandas as pd
import os
import numpy as np
from pathlib import Path
from ultralytics import YOLO
from sklearn.metrics import confusion_matrix, classification_report, cohen_kappa_score, accuracy_score, precision_recall_fscore_support

from utils import *

import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

hola


In [ ]:
BASE_PATH = Path("../data")
EXP_PATH = BASE_PATH / "EXPERIMENT1"
DOMAIN_PATH = EXP_PATH / "domain_adaptation"

TEST_PUBLIC_PATH = DOMAIN_PATH / "test"   # In-domain test
TEST_LOCAL_PATH  = EXP_PATH / "local_data"          # Out-of-domain test
MODELS_PATH = Path("../models")

BASE_MODELS_PATH = MODELS_PATH / "base_models"
FT_MODELS_PATH = MODELS_PATH / "finetuned_models"
OUTPUT_PATH = Path("../results")


FIGS_DIRECTORY = BASE_PATH / "paper_figs"
os.makedirs(FIGS_DIRECTORY, exist_ok=True)

BATCH_SIZE = 32

IMG_SIZE = (224,224)
model_list = [
    "efficientnetb0_CBAM",
    "mobilenetv2_CBAM",
    "efficientnetb3_CBAM",
    "resnet101_CBAM",
    "mobilenetv2",
    "efficientnetb0",
    "mobilenetv3large",
    "densenet121",
    "efficientnetb3",
    "resnet50",
    "inceptionv3",
    "resnet101",
    "vgg16",
    "vgg19",
    "efficientnetb5"
]

NUM_CLASSES = 4

# Evaluating CNN models upon known domain (public test)

In [ ]:
TEST_PATH = Path(r"../data/dataset/test")

CONF_MATRIX_DIR = BASE_MODELS_PATH / "confusion_matrix"

## Model evaluation (base models and CBAM)

In [ ]:
CLASSES = sorted([d.name for d in TEST_PATH.iterdir() if d.is_dir()])
class_indices = {cls: i for i, cls in enumerate(CLASSES)}
model_list = [
    "mobilenetv2", "efficientnetb0", "efficientnetb3", "mobilenetv3large", "densenet121",
    "resnet50", "resnet101", "Inceptionv3", "vgg16", "vgg19", "efficientnetb5"
]


In [ ]:
for backbone_name in model_list:
    print(f"\n=== EVALUATING: {backbone_name} ===\n")

    best_model_path = BASE_MODELS_PATH / f"{backbone_name}_best_model.keras"
    if not best_model_path.exists():
        print(f"Model: {backbone_name} not found, skipping.")
        continue

    model = tf.keras.models.load_model(best_model_path, compile=False)

    X_list, y_list = [], []

    for cls in CLASSES:
        class_dir = TEST_PATH / cls
        if not class_dir.exists():
            print(f"Absent class: {cls}")
            continue

        imgs = list(class_dir.glob("*.jpg")) + list(class_dir.glob("*.png")) + list(class_dir.glob("*.jpeg"))

        for img_path in imgs:
            img = cv2.imread(str(img_path))
            if img is None:
                continue

            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, IMG_SIZE)

            img = img.astype(np.float32) / 255.0
            img = preprocess_for_backbone(img, backbone_name) 

            X_list.append(img)
            y_list.append(class_indices[cls])

    if not X_list:
        print(f"Not available images for model {backbone_name}.")
        continue

    X_test = np.stack(X_list, axis=0)
    y_true = np.array(y_list)

    preds = model.predict(X_test, batch_size=BATCH_SIZE, verbose=1)
    y_pred = np.argmax(preds, axis=1)

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    kappa = cohen_kappa_score(y_true, y_pred)

    print(f"\n=== GLOBAL METRICS FOR {backbone_name} ===")
    print(f"Accuracy:       {accuracy:.4f}")
    print(f"Precision:      {precision:.4f}")
    print(f"Recall:         {recall:.4f}")
    print(f"F1-score:       {f1:.4f}")
    print(f"Cohen's Kappa:  {kappa:.4f}\n")

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=np.arange(len(CLASSES))
    )

    cm_color = cm.astype(np.float32) / cm.sum(axis=1, keepdims=True)
    cm_color = np.nan_to_num(cm_color)

    plt.figure(figsize=(8, 6)) 

    sns.heatmap(
        cm_color,          
        annot=cm,          
        fmt="d",
        cmap="Blues",
        xticklabels=CLASSES,
        yticklabels=CLASSES,
        vmin=0, vmax=1
    )

    plt.xlabel("Predicted")
    plt.ylabel("Real")


    plt.xticks(rotation=45, ha="right")
    plt.tight_layout() 
    out_path = CONF_MATRIX_DIR / f"confusion_matrix_{backbone_name}.png"
    plt.savefig(out_path, dpi=300)


    plt.show()
    plt.close()

    print(f"Confusion matrix saved to: {out_path}")

## Comparison of models CBAM vs base ones.

In [ ]:
experiments = [
    {"name": "efficientnetb0_CBAM", "backbone": "efficientnetb0"},
    {"name": "mobilenetv2_CBAM",    "backbone": "mobilenetv2"},
    {"name": "efficientnetb3_CBAM", "backbone": "efficientnetb3"},
    {"name": "resnet101_CBAM",  "backbone": "resnet101"},
]

In [ ]:
colors = ["#1f77b4", "#ff7f0e"]

def plot_model_metrics(df, model_name, classes, save_path=None):
    sub = df[df["Model"] == model_name]
    base = sub[sub["Variant"]=="base"].iloc[0]
    cbam = sub[sub["Variant"]=="cbam"].iloc[0]

    f1_base = np.array(base["F1_per_class"], dtype=float)
    f1_cbam = np.array(cbam["F1_per_class"], dtype=float)
    prec_base = np.array(base["Precision_per_class"], dtype=float)
    prec_cbam = np.array(cbam["Precision_per_class"], dtype=float)
    rec_base = np.array(base["Recall_per_class"], dtype=float)
    rec_cbam = np.array(cbam["Recall_per_class"], dtype=float)

    x = np.arange(len(classes))
    width = 0.35

    fig, ax = plt.subplots(1, 3, figsize=(13,4))
    fig.subplots_adjust(wspace=0.3)

    # --- F1 ---
    ax[0].bar(x - width/2, f1_base, width, label="Base", color=colors[0])
    ax[0].bar(x + width/2, f1_cbam, width, label="CBAM", color=colors[1])
    ax[0].set_xticks(x)
    ax[0].set_xticklabels(classes, rotation=45, ha="right")
    ax[0].set_ylim(0, 1)
    ax[0].set_title("F1 Score")

    # --- Precision ---
    ax[1].bar(x - width/2, prec_base, width, label="Base", color=colors[0])
    ax[1].bar(x + width/2, prec_cbam, width, label="CBAM", color=colors[1])
    ax[1].set_xticks(x)
    ax[1].set_xticklabels(classes, rotation=45, ha="right")
    ax[1].set_ylim(0, 1)
    ax[1].set_title("Precision")

    # --- Recall ---
    ax[2].bar(x - width/2, rec_base, width, label="Base", color=colors[0])
    ax[2].bar(x + width/2, rec_cbam, width, label="CBAM", color=colors[1])
    ax[2].set_xticks(x)
    ax[2].set_xticklabels(classes, rotation=45, ha="right")
    ax[2].set_ylim(0, 1)
    ax[2].set_title("Recall")


    ax[2].legend(loc='upper right')

    sns.despine()
    plt.tight_layout()
    plt.savefig(save_path)
    plt.show()
 

In [ ]:
resumen_detallado = []
CLASSES = sorted([d.name for d in TEST_PATH.iterdir() if d.is_dir()])
class_indices = {cls: i for i, cls in enumerate(CLASSES)}

for exp in experiments:
    print(f"\n" + "="*50)
    print(f"Evaluating: {exp['name']}")
    print("="*50)

    best_model_path = BASE_MODELS_PATH / f"{exp['name']}_best_model.keras"
    if not best_model_path.exists():
        print(f"Model not found: {best_model_path.name}")
        continue

    custom_objects = {
        'SpatialPooling': SpatialPooling,
        'f1_macro': f1_macro,
        'precision_macro': precision_macro,
        'recall_macro': recall_macro
    }
    model = tf.keras.models.load_model(best_model_path, custom_objects=custom_objects, compile=False)

    X_list, y_list = [], []
    for cls in CLASSES:
        class_dir = TEST_PATH / cls
        if not class_dir.exists(): continue

        imgs = list(class_dir.glob("*.jpg")) + list(class_dir.glob("*.png"))
        for img_path in imgs:
            img = cv2.imread(str(img_path))
            if img is None: continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, IMG_SIZE)
            img = img.astype(np.float32) / 255.0
            img = preprocess_for_backbone(img, exp['backbone'])
            X_list.append(img)
            y_list.append(class_indices[cls])

    if not X_list:
        print(f"Test images not found for {exp['name']}")
        continue

    X_test = np.stack(X_list, axis=0)
    y_true = np.array(y_list)

    # Preds
    preds = model.predict(X_test, batch_size=BATCH_SIZE, verbose=1)
    y_pred = np.argmax(preds, axis=1)

    prec_pc, rec_pc, f1_pc, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=np.arange(len(CLASSES)), zero_division=0
    )

    resumen_detallado.append({
        'Model': exp['backbone'],     
        'Variant': 'cbam' if exp['use_cbam'] else 'base',
        'Precision_per_class': prec_pc,
        'Recall_per_class': rec_pc,
        'F1_per_class': f1_pc,
        'Accuracy': accuracy_score(y_true, y_pred),
        'F1_Weighted': f1_score(y_true, y_pred, average='weighted', zero_division=0)
    })

    # confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(CLASSES)))
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=CLASSES, yticklabels=CLASSES)
    plt.xlabel("Predicted")
    plt.ylabel("Real")
    plt.title(f"Confusion Matrix: {exp['name']}")
    out_path = CONF_MATRIX_DIR / f"cm_{exp['name']}.png"
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close()

df = pd.DataFrame(resumen_detallado)

print(df[["Model", "Variant", "Accuracy", "F1_Weighted"]])

for model_name in df["Model"].unique():
    print(f"\n==================\nModel: {model_name}\n==================\n")
    plot_model_metrics(df, model_name, CLASSES,save_path=FIGS_DIRECTORY / f"{model_name}_metrics.png")

# Evaluating YOLO models upon known domain (public test)

In [ ]:
DATASET_TEST = Path("..data/dataset/dataset_YOLO/test")
PROJECT_DIR_PATH = Path("..data/models/results")

In [ ]:
MODEL_VARIANTS = ["n", "m", "l"]
RESULTS_SUMMARY = [] 

class_dirs = sorted([d.name for d in DATASET_TEST.iterdir() if d.is_dir()])
file_list = []
y_true = []

for idx, name in enumerate(class_dirs):
    images = [str(f) for f in (DATASET_TEST / name).rglob("*") if f.suffix.lower() in [".jpg", ".jpeg", ".png"]]
    file_list.extend(images)
    y_true.extend([idx] * len(images))

print(f"Data is ready: {len(file_list)} images, {len(class_dirs)} classes.")

for v in MODEL_VARIANTS:
    model_path = PROJECT_DIR_PATH / f"YOLO11{v}_cls" / "weights" / "best.pt"
    
    if not model_path.exists():
        print(f"Skipping YOLO11{v}: File not found in {model_path}")
        continue

    print(f"\n🚀 Evaluating YOLO11{v}...")
    model = YOLO(model_path)
    
    start_time = time.time()
    results = model.predict(source=file_list, device=0, verbose=False)
    inference_time = (time.time() - start_time) / len(file_list) 
    
    y_pred = [r.probs.top1 for r in results]
    

    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted')
    kappa = cohen_kappa_score(y_true, y_pred)
    
    RESULTS_SUMMARY.append({
        "Model": f"YOLO11{v}",
        "Acc": acc,
        "F1-Score": f1,
        "Kappa": kappa,
        "Time/Img (s)": inference_time
    })
    

    print(f"--- Report {v.upper()} ---")
    print(classification_report(y_true, y_pred, target_names=class_dirs))

print("\n" + "="*60 + "\n📊 Model comparison\n" + "="*60)
df_results = pd.DataFrame(RESULTS_SUMMARY).set_index("Model")
print(df_results.round(4))

# EVALUATING DOMAIN GAP 

## FOR CNN-MODELS

In [ ]:
results = []

class_weights_array_for_loading = np.ones(NUM_CLASSES)

for backbone in model_list:
    print(f"\n🔹 Evaluating: {backbone}")

    model = None 
    best_model_path = BASE_MODELS_PATH / f"{backbone}_best_model.keras"
    if not best_model_path.exists():
        print(f"Model not found: {best_model_path}")
        continue 

    custom_objects = {
    "weighted_categorical_crossentropy": weighted_categorical_crossentropy,
    "recall_macro": recall_macro,
    "precision_macro": precision_macro,
    "f1_macro": f1_macro
    }  

    model = tf.keras.models.load_model(
        best_model_path,
        compile=False,
        custom_objects=custom_objects
    )


    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss=weighted_categorical_crossentropy(class_weights_array_for_loading),
        metrics=[f1_macro, precision_macro, recall_macro]
    )

    # Determine the correct backbone name for the generator
    gen_backbone_name = backbone.lower().replace("_cbam", "")

    gen_public = AlbumentationsDataGenerator(TEST_PUBLIC_PATH, batch_size=BATCH_SIZE, img_size=IMG_SIZE, backbone=gen_backbone_name)
    gen_local  = AlbumentationsDataGenerator(TEST_LOCAL_PATH,  batch_size=BATCH_SIZE, img_size=IMG_SIZE, backbone=gen_backbone_name)

    y_true_public = get_true_labels_from_keras_generator(gen_public)
    y_true_local  = get_true_labels_from_keras_generator(gen_local)

    y_pred_public = np.argmax(model.predict(gen_public, verbose=0), axis=1)
    y_pred_local  = np.argmax(model.predict(gen_local, verbose=0), axis=1)

    # Ensure model is not None (safety check)
    if model is None:
        print(f"Error: Model '{backbone}' could not be loaded. Skipping metrics")
        continue

    # Metrics
    report_public = classification_report(y_true_public, y_pred_public, output_dict=True, zero_division=0)
    report_local  = classification_report(y_true_local, y_pred_local, output_dict=True, zero_division=0)

    acc_public = report_public["accuracy"]
    acc_local  = report_local["accuracy"]
    f1_public  = report_public["weighted avg"]["f1-score"]
    f1_local   = report_local["weighted avg"]["f1-score"]

    domain_gap_acc = acc_public - acc_local
    domain_gap_f1  = f1_public - f1_local

    results.append({
        "Model": backbone,
        "Accuracy_Public": acc_public,
        "Accuracy_Local": acc_local,
        "F1_Public": f1_public,
        "F1_Local": f1_local,
        "Δ Accuracy": domain_gap_acc,
        "Δ F1": domain_gap_f1,
    })

    # confusion matrix
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, (y_true, y_pred, title) in zip(
        axes,
        [(y_true_public, y_pred_public, "Public Test"),
         (y_true_local, y_pred_local, "Local Test")]
    ):
        cm = confusion_matrix(y_true, y_pred)
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
        ax.set_title(f"{backbone} - {title}")
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")
    plt.tight_layout()
    plt.show()

# save results
df_results = pd.DataFrame(results).sort_values("Δ F1", ascending=False)
df_results.to_csv(OUTPUT_PATH / "domain_gap_metrics.csv", index=False)
display(df_results)

# Δ F1 Plot
plt.figure(figsize=(10,6))
plt.bar(df_results["Model"], df_results["Δ F1"], color="salmon")
plt.title("Domain Gap (Δ F1)")
plt.ylabel("Δ F1-score (Public - Local)")
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## YOLO MODELS

In [ ]:
datasets = {
    "Public": TEST_PUBLIC_PATH,
    "Local": TEST_LOCAL_PATH
}

models_info = {
    "YOLO11n": BASE_MODELS_PATH / "YOLO11n_cls" / "weights" / "best.pt",
    "YOLO11m": BASE_MODELS_PATH / "YOLO11m_cls" / "weights" / "best.pt",
    "YOLO11l": BASE_MODELS_PATH / "YOLO11l_cls" / "weights" / "best.pt",
}

all_results = []

for model_name, model_path in models_info.items():
    if not model_path.exists():
        print(f"⚠️ Model not found: {model_path}")
        continue

    print(f"\n" + "="*50 + f"\n🚀 Evaluating Model: {model_name}\n" + "="*50)
    model = YOLO(model_path)
    
    model_metrics = {"Model": model_name}
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    for idx, (ds_name, ds_path) in enumerate(datasets.items()):
        print(f"🧐 Processing {ds_name} Dataset...")
        
        class_dirs = sorted([d.name for d in ds_path.iterdir() if d.is_dir()])
        file_list = []
        y_true = []
        
        for class_idx, name in enumerate(class_dirs):
            images = [str(f) for f in (ds_path / name).rglob("*") if f.suffix.lower() in [".jpg", ".jpeg", ".png"]]
            file_list.extend(images)
            y_true.extend([class_idx] * len(images))
        
        results = model.predict(source=file_list, imgsz=224, device=0, verbose=False)
        y_pred = [r.probs.top1 for r in results]

        report = classification_report(y_true, y_pred, target_names=class_dirs, output_dict=True, zero_division=0)
        acc = accuracy_score(y_true, y_pred)
        f1_weighted = report["weighted avg"]["f1-score"]
        
        model_metrics[f"Accuracy_{ds_name}"] = acc
        model_metrics[f"F1_{ds_name}"] = f1_weighted
        
        cm = confusion_matrix(y_true, y_pred)
        sns.heatmap(cm, annot=True, fmt="d", cmap="Greens" if ds_name == "Local" else "Blues", 
                    ax=axes[idx], xticklabels=class_dirs, yticklabels=class_dirs)
        axes[idx].set_title(f"{model_name} - {ds_name} Domain")
        axes[idx].set_xlabel("Predicted Label")
        axes[idx].set_ylabel("True Label")

    model_metrics["Δ_Accuracy"] = model_metrics["Accuracy_Public"] - model_metrics["Accuracy_Local"]
    model_metrics["Δ_F1"] = model_metrics["F1_Public"] - model_metrics["F1_Local"]
    
    all_results.append(model_metrics)
    plt.tight_layout()
    plt.show()

df_results = pd.DataFrame(all_results).sort_values("Δ_F1", ascending=False)
csv_filename = OUTPUT_PATH / "domain_gap_YOLOorig.csv"
df_results.to_csv(csv_filename, index=False)

print("\n" + "="*60 + f"\n✅ Evaluation Complete. Summary saved to:\n{csv_filename}\n" + "="*60)
display(df_results.round(4))

# Evaluating domain gap after Transfer Learning.

## All strategies considered (ESTO QUIZÁ LO QUITEMOS TRAS HACER EL GRID Y ESO, PUES YA SE HABRÁ SELECCIONADO EL MEJOR MODELO) --  Table 7 paper

###  CNN models. 

In [ ]:
strategies = ["feature_extractor", "fine_tune_last", "fine_tune_all"]
evaluation_rows = []

for backbone in model_list:
    for strategy in strategies:

        # Paths
        candidate_strategy = FT_MODELS_PATH/ strategy / f"{backbone}_best_{strategy}.keras"
        candidate_base     = FT_MODELS_PATH / strategy / f"{backbone}_best_model.keras"

        if candidate_strategy.exists():
            model_type = "fine_tuned"
            model_path = candidate_strategy
        elif candidate_base.exists():
            model_type = "base"
            model_path = candidate_base
        else:
            print(f"Model not found for: {backbone} | {strategy}")
            continue

        print(f"\n Evaluating: {backbone} | {strategy} ({model_type})")
        print(f"Path: {model_path}")
        if backbone.endswith("CBAM"):
            alb_backbone = backbone.lower().replace("_cbam", "")
        else:
            alb_backbone = backbone.lower()
        # test gen
        test_gen = AlbumentationsDataGenerator(
            TEST_LOCAL_PATH,
            batch_size=16,
            img_size=IMG_SIZE,
            oversample=False,
            augment=False,
            backbone=alb_backbone
        )

        NUM_CLASSES = len(test_gen.classes)

        custom_objects = {
        "weighted_categorical_crossentropy": weighted_categorical_crossentropy,
        "recall_macro": recall_macro,
        "precision_macro": precision_macro,
        "f1_macro": f1_macro
        }  
        try:
            model = load_model(
                model_path,
                compile=False,
                custom_objects=custom_objects
            )
        except Exception as e:
            print(f"Could not load model {model_path}: {e}")
            continue

        # eval
        y_true, y_pred = [], []
        for i in range(len(test_gen)):
            Xb, yb = test_gen[i]
            preds = model.predict(Xb, verbose=0)
            y_true.extend(np.argmax(yb, axis=1))
            y_pred.extend(np.argmax(preds, axis=1))

        y_true = np.array(y_true)
        y_pred = np.array(y_pred)

        acc = accuracy_score(y_true, y_pred)
        f1  = f1_score(y_true, y_pred, average="weighted")

        # conf matrix
        class_names = test_gen.classes
        cm = confusion_matrix(y_true, y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                    xticklabels=class_names, yticklabels=class_names)
        plt.title(f"Confusion Matrix — {backbone} | {strategy}")
        plt.xlabel("Predicted")
        plt.ylabel("True")
        png_path = CONF_MATRIX_DIR / f"cm_{backbone}_{strategy}.png"
        plt.tight_layout()
        plt.savefig(png_path, dpi=150)
        plt.close()

        # save
        evaluation_rows.append({
            "model": backbone,
            "strategy": strategy,
            "f1_test": f1,
            "accuracy_test": acc,
            "saved_model": str(model_path),
            "confusion_matrix_png": str(png_path)
        })

# final CSV
if evaluation_rows:
    df_eval = pd.DataFrame(evaluation_rows)
    CSV_RESULTS = EXP_PATH / "results" / "all_models_evaluation.csv"
    df_eval.to_csv(CSV_RESULTS, index=False)
    print("\n✅ CSV saved:", CSV_RESULTS)
    display(df_eval.sort_values("f1_test", ascending=False))
else:
    print("Models not found. Check paths and custom_objects.")

### YOLO models

In [ ]:
# Define models and strategies used in training
model_names = ["yolov11n", "yolov11m", "yolov11l"]
strategies = ["feature_extractor", "fine_tune_last", "fine_tune_all"]

evaluation_rows = []

class_dirs = sorted([d.name for d in TEST_LOCAL_PATH.iterdir() if d.is_dir()])
file_list = []
y_true = []

for idx, class_name in enumerate(class_dirs):
    images = [str(f) for f in (TEST_LOCAL_PATH / class_name).rglob("*") 
              if f.suffix.lower() in [".jpg", ".jpeg", ".png"]]
    file_list.extend(images)
    y_true.extend([idx] * len(images))

print(f"✅ Data ready: {len(file_list)} images found across {len(class_dirs)} classes.")

for strategy in strategies:
    print(f"\n" + "="*50 + f"\n📂 Strategy: {strategy.upper()}\n" + "="*50)
    
    for m_name in model_names:
        model_path = FT_MODELS_PATH / strategy / m_name / "weights" / "best.pt"
        
        if not model_path.exists():
            print(f"⚠️ Skipping {m_name} | {strategy}: Model not found at {model_path}")
            continue

        print(f"🚀 Evaluating: {m_name} | Strategy: {strategy}")
        
        model = YOLO(model_path)
        results = model.predict(source=file_list, imgsz=224, device=0, verbose=False)
        y_pred = [r.probs.top1 for r in results]
        
        # Metrics
        acc = accuracy_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred, average="weighted")
        
        # Confusion Matrix
        cm = confusion_matrix(y_true, y_pred)
        plt.figure(figsize=(10, 8))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", 
                    xticklabels=class_dirs, yticklabels=class_dirs)
        plt.title(f"CM: {m_name} - {strategy}")
        plt.xlabel("Predicted")
        plt.ylabel("True")
        
        cm_path = CONF_MATRIX_DIR / f"cm_yolo_{m_name}_{strategy}.png"
        plt.tight_layout()
        plt.savefig(cm_path, dpi=150)
        plt.close()
        
        evaluation_rows.append({
            "model": m_name,
            "strategy": strategy,
            "accuracy_test": acc,
            "f1_test": f1,
            "model_path": str(model_path),
            "cm_path": str(cm_path)
        })

if evaluation_rows:
    df_eval = pd.DataFrame(evaluation_rows)
    csv_path = OUTPUT_PATH / "yolo_finetuning_evaluation.csv"
    df_eval.to_csv(csv_path, index=False)
    
    print(f"\n✅ Evaluation finished! CSV saved at: {csv_path}")
    display(df_eval.sort_values(by="f1_test", ascending=False))
else:
    print("❌ No evaluation data collected. Check your model paths.")

## BEST STRATEGY FOR EACH MODEL BEING EVALUATED (TABLE 8 paper)

### CNN models

In [ ]:
# DOMAIN GAP - CONSIDERING BEST STRATEGY

results = []

# load csv
csv_unified = EXP_PATH / "results" / "total_models_evaluation.csv"
df_all = pd.read_csv(csv_unified)

# best strategy (criterion: F1)
df_best = df_all.loc[df_all.groupby("model")["f1_test"].idxmax()]

all_models_best = df_best["model"].tolist()


# loop for eval

for backbone in all_models_best:
    print(f"\n🔹 Evaluando modelo: {backbone}")



    strategy = df_best.loc[df_best["model"] == backbone, "strategy"].values[0]
    model_path = FT_MODELS_PATH / strategy / f"{backbone}_{strategy}.keras"
    if not model_path.exists():
        print(f"Model not found: {model_path}")
        continue
    model = tf.keras.models.load_model(model_path, custom_objects=custom_objects)

    gen_public = AlbumentationsDataGenerator(TEST_PUBLIC_PATH, batch_size=BATCH_SIZE, img_size=IMG_SIZE, backbone=backbone)
    gen_local  = AlbumentationsDataGenerator(EXP_PATH / "test",  batch_size=BATCH_SIZE, img_size=IMG_SIZE, backbone=backbone)

    y_true_public = get_true_labels_from_keras_generator(gen_public)
    y_true_local  = get_true_labels_from_keras_generator(gen_local)
    y_pred_public = np.argmax(model.predict(gen_public, verbose=0), axis=1)
    y_pred_local  = np.argmax(model.predict(gen_local, verbose=0), axis=1)


    # Metrics and ΔF1

    report_public = classification_report(y_true_public, y_pred_public, output_dict=True, zero_division=0)
    report_local  = classification_report(y_true_local, y_pred_local, output_dict=True, zero_division=0)

    acc_public = report_public["accuracy"]
    acc_local  = report_local["accuracy"]
    f1_public  = report_public["weighted avg"]["f1-score"]
    f1_local   = report_local["weighted avg"]["f1-score"]

    domain_gap_acc = acc_public - acc_local
    domain_gap_f1  = f1_public - f1_local

    results.append({
        "Model": backbone,
        "Strategy": strategy,
        "Accuracy_Public": acc_public,
        "Accuracy_Local": acc_local,
        "F1_Public": f1_public,
        "F1_Local": f1_local,
        "Δ Accuracy": domain_gap_acc,
        "Δ F1": domain_gap_f1,
    })

    # conf matrix
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, (y_t, y_p, title) in zip(axes, [(y_true_public, y_pred_public, "Public Test"), (y_true_local, y_pred_local, "Local Test")]):
        cm = confusion_matrix(y_t, y_p)
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
        ax.set_title(f"{backbone} - {title}")
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")
    plt.tight_layout()
    plt.show()

# save results
df_results = pd.DataFrame(results).sort_values("Δ F1", ascending=False)
df_results.to_csv(OUTPUT_PATH / "domain_gap_CNN_best_strategy.csv", index=False)
display(df_results)

# Δ F1 plot
plt.figure(figsize=(10,6))
plt.bar(df_results["Model"], df_results["Δ F1"], color="salmon")
plt.title("Domain Gap (Δ F1) between Public and  Localdata  — Best strategy only")
plt.ylabel("Δ F1-score")
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### YOLO models

In [ ]:
CSV_YOLO_RESULTS = EXP_PATH / "results" / "yolo_finetuning_evaluation.csv"

OUTPUT_CSV = OUTPUT_PATH / "domain_gap_yolo_best_strategy.csv"

# Domain Datasets
DATASETS = {
    "Public": TEST_PUBLIC_PATH,
    "Local": TEST_LOCAL_PATH
}

if not CSV_YOLO_RESULTS.exists():
    raise FileNotFoundError(f"Could not find {CSV_YOLO_RESULTS}. Run the evaluation script first.")

df_all = pd.read_csv(CSV_YOLO_RESULTS)

df_best = df_all.loc[df_all.groupby("model")["f1_test"].idxmax()]

print("="*60)
print("EVALUATING DOMAIN GAP FOR BEST STRATEGIES")
print("="*60)

domain_results = []

for _, row in df_best.iterrows():
    model_name = row['model']
    strategy = row['strategy']
    model_path = Path(row['model_path']) 

    if not model_path.exists():
        print(f"⚠️ Model path not found: {model_path}")
        continue

    print(f"\n🔹 Model: {model_name} | Best Strategy: {strategy}")
    model = YOLO(model_path)
    
    metrics_per_domain = {
        "Model": model_name,
        "Best_Strategy": strategy
    }

    for domain_name, domain_path in DATASETS.items():
        print(f"   Processing {domain_name} domain...")

        class_dirs = sorted([d.name for d in domain_path.iterdir() if d.is_dir()])
        file_list = []
        y_true = []
        
        for idx, c_name in enumerate(class_dirs):
            imgs = [str(f) for f in (domain_path / c_name).rglob("*") if f.suffix.lower() in [".jpg", ".jpeg", ".png"]]
            file_list.extend(imgs)
            y_true.extend([idx] * len(imgs))

        preds = model.predict(source=file_list, imgsz=224, device=0, verbose=False)
        y_pred = [r.probs.top1 for r in preds]

        acc = accuracy_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred, average="weighted")
        
        metrics_per_domain[f"Accuracy_{domain_name}"] = acc
        metrics_per_domain[f"F1_{domain_name}"] = f1

    metrics_per_domain["Δ_Accuracy"] = metrics_per_domain["Accuracy_Public"] - metrics_per_domain["Accuracy_Local"]
    metrics_per_domain["Δ_F1"] = metrics_per_domain["F1_Public"] - metrics_per_domain["F1_Local"]
    
    domain_results.append(metrics_per_domain)


df_domain_gap = pd.DataFrame(domain_results).sort_values("Δ_F1", ascending=False)
df_domain_gap.to_csv(OUTPUT_CSV, index=False)

print("\n" + "="*60)
print(f"✅ Domain Gap analysis saved to: {OUTPUT_CSV}")
print("="*60)
display(df_domain_gap.round(4))

# DOMAIN GAP COMPARISON (figure 7 paper)

Unify domain gap before Transfer Learning:

In [ ]:
CSV_KERAS = OUTPUT_PATH / "domain_gap_metrics.csv"       # CSV Keras
CSV_YOLO  = OUTPUT_PATH / "domain_gap_YOLOorig.csv"      # CSV YOLO
CSV_UNIFIED = OUTPUT_PATH / "domain_gap_baseline_all_models.csv"  # unified CSV

df_keras = pd.read_csv(CSV_KERAS)
df_yolo  = pd.read_csv(CSV_YOLO)

df_all = pd.concat([df_keras, df_yolo], ignore_index=True)

df_all.to_csv(CSV_UNIFIED, index=False)
print("CSV saved:", CSV_UNIFIED)
display(df_all)

Unify domain gap after Transfer Learning:

In [ ]:
CSV_KERAS = OUTPUT_PATH / "domain_gap_CNN_best_strategy.csv"
CSV_YOLO  =  OUTPUT_PATH / "domain_gap_yolo_best_strategy.csv"
CSV_UNIFIED = OUTPUT_PATH/ "domain_gap_all_models_best_strategy.csv"

df_keras = pd.read_csv(CSV_KERAS)
df_yolo  = pd.read_csv(CSV_YOLO)

df_all = pd.concat([df_keras, df_yolo], ignore_index=True)

df_all.to_csv(CSV_UNIFIED, index=False)
print("✅ CSV unificado guardado en:", CSV_UNIFIED)
display(df_all)

Convert to df the csv we want to compare:

In [ ]:
csv_ft = OUTPUT_PATH / "domain_gap_all_models_best_strategy.csv"
csv_original = OUTPUT_PATH / "domain_gap_baseline_all_models.csv"

df_orig = pd.read_csv(csv_original)
df_ft = pd.read_csv(csv_ft)

In [ ]:
display(df_orig.head())
display(df_ft.head())

Creating the plot for the paper (figure 7) which compares domain gap before/after FT:

In [ ]:
# normalizing names for the merge

def normalize_model(x):
    return x.lower().replace("+","").replace(" ","").replace("_gap","")

df_orig["model_norm"] = df_orig["Model"].apply(normalize_model)
df_ft["model_norm"] = df_ft["Model"].apply(normalize_model)

# merge for common models (inner)
df = df_orig.merge(df_ft, on="model_norm", how="inner", suffixes=("_before", "_after"))

# compute: ΔF1 = F1_Public − F1_Local

df["DeltaF1_Before"] = df["F1_Public_before"] - df["F1_Local_before"]
df["DeltaF1_After"]  = df["F1_Public_after"] - df["F1_Local_after"]

# preparing for sns
df_plot = pd.melt(
    df,
    id_vars=["Model_before"],
    value_vars=["DeltaF1_Before", "DeltaF1_After"],
    var_name="Condition",
    value_name="DeltaF1"
)

df_plot["Condition"] = df_plot["Condition"].replace({
    "DeltaF1_Before": "Before FT",
    "DeltaF1_After": "After FT"
})

# seaborn plot
sns.set_theme(style="whitegrid", font_scale=1.3)
plt.figure(figsize=(14,6))
palette = sns.color_palette("viridis", 2)

ax = sns.barplot(
    data=df_plot,
    x="Model_before", y="DeltaF1", hue="Condition",
    palette=palette, edgecolor="black", linewidth=1.1
)

plt.xticks(rotation=35, ha="right")
plt.ylabel("ΔF1")
plt.xlabel("")

plt.axhline(0, color="black", linewidth=1, linestyle="--", alpha=0.7)

# adding values upon bars, slightly above
offset = 0.003

for p in ax.patches:
    height = p.get_height()

    if abs(height) < 1e-6:
        continue

    ax.annotate(
        f"{height:.3f}",
        (p.get_x() + p.get_width() / 2, height + offset),
        ha='center',
        va='bottom',
        fontsize=10,
        rotation=90
    )


plt.tight_layout()
plt.show()